# IFT6390 — Compétition 2 (2026) | Notebook final

**Auteur :** Assadiki Mohamed Yassine · **Matricule :** 20286183  
**Équipe :** `ift6390_mohamedyassine_assadiki`  
**Meilleure soumission publique historique :** **0,02808** (`submission_v10_unet_geometry.csv`).

### L'idée de V10
1. Des images de 64 × 64 pixels sont perturbées par un bruit gaussien approximatif de σ = 0,133942 et des translations simulées jusqu'à ±10 pixels.
2. Un **U-Net créé et entraîné depuis zéro** débruite les images ; aucune donnée ou modèle externe n'est utilisé.
3. Le régresseur **HistGradientBoostingRegressor** apprend sur **70 variables** (24 radiales, 18 géométriques, 14 lissées, 14 débruitées).
4. **ExtraTreesRegressor** apprend sur les **42 premières variables**. La prédiction finale est `0,70 * boosting70 + 0,30 * extra42`.


### Structure des fichiers
```text
Projet/
├── notebook_final_V10_reproduction_corrigee_v2.ipynb
├── data/
│   ├── train_images.npy
│   ├── train_labels.csv
│   ├── test_images.npy
│   └── sample_submission.csv                 # optionnel
├── results/
│   ├── denoiser_full_v10.pt
│   ├── boosting_v10_unet_geometry.joblib
│   ├── boosting_v8_smoothed.joblib           # si disponible : restauration V8
│   ├── boosting_v4_geometry.joblib           # restauration alternative V7
│   └── extra_trees_v7.joblib                # peut être illisible
└── submissions/
    ├── submission_v8_smoothed_blend.csv      # si disponible
    ├── submission_v7_boosting_extratrees.csv # alternative
    └── submission_v10_unet_geometry.csv      # original : AUDIT uniquement
```



In [1]:
# 01 — Dépendances, chemins de travail, graines aléatoires et hyperparamètres
# Les bibliothèques utilisées sont générales ; aucun modèle préentraîné externe n'est chargé.
from pathlib import Path
import os
import random
import time
import copy
import gc
import json
import hashlib
import warnings

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
from scipy.ndimage import gaussian_filter
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.ensemble import HistGradientBoostingRegressor, ExtraTreesRegressor
from sklearn.metrics import mean_absolute_error
import sklearn
import joblib

# Le notebook doit se trouver à la racine du projet ; VS Code utilise le répertoire courant.
DATA_DIR = Path("data")
RESULTS_DIR = Path("results")
SUBMISSIONS_DIR = Path("submissions")
for directory in (RESULTS_DIR, SUBMISSIONS_DIR):
    directory.mkdir(parents=True, exist_ok=True)

# Graines de l'expérimentation originale.
SEED = 42
AUG_SEED = 2026

# Décalage train/test observé : images test déjà bruitées et parfois translatées.
NOISE_SIGMA = 0.133942
INTENSITY_SCALE = 0.2  # arcsinh(x / 0.2) comprime les intensités extrêmes.
MAX_SHIFT = 10          # Translation maximale, en pixels, dans chaque axe.
N_COPIES = 5            # Cinq versions de 2000 images = 10000 exemples régresseur.

# Options utilisateur : en cas de chargement impossible, régénération contrôlée.
USE_SAVED_MODELS = True
RUN_CV = False
RETRAIN_CV_DENOISERS_IF_MISSING = False
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Fixer les générateurs aléatoires limite la variabilité entre les exécutions.
np.random.seed(SEED)
random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# Régression L1 : optimisée pour le critère MAE de la compétition.
HGB_PARAMS = dict(
    loss="absolute_error", learning_rate=0.05, max_iter=200,
    max_leaf_nodes=15, min_samples_leaf=20,
    l2_regularization=1.0, early_stopping=False, random_state=42
)
# Arbres extrêmement aléatoires : prédictions complémentaires au Boosting.
EXTRA_PARAMS = dict(
    n_estimators=250, min_samples_leaf=2, max_features=0.85,
    random_state=42, n_jobs=-1
)

print("Environnement :", "Python/PyTorch", torch.__version__, "| GPU :", DEVICE)
print("NumPy :", np.__version__, "| Scikit-learn :", sklearn.__version__)
print("Joblib :", joblib.__version__, "| Dossier de travail :", Path.cwd())


Environnement : Python/PyTorch 2.15.0+cpu | GPU : cpu
NumPy : 2.5.5 | Scikit-learn : 1.9.1
Joblib : 1.6.0 | Dossier de travail : c:\Users\yassi\Desktop\IFT_COMP2


In [2]:
# 02. Charger et vérifier les fichiers de la compétition
required = [DATA_DIR / "train_images.npy", DATA_DIR / "train_labels.csv",
            DATA_DIR / "test_images.npy"]
missing = [str(p) for p in required if not p.exists()]
if missing:
    raise FileNotFoundError("Placer les fichiers Kaggle dans data/ : " + ", ".join(missing))

# float32 conserve le format des pixels prévu par la compétition.
X_train = np.load(DATA_DIR / "train_images.npy").astype(np.float32)
X_test = np.load(DATA_DIR / "test_images.npy").astype(np.float32)
labels = pd.read_csv(DATA_DIR / "train_labels.csv")
assert X_train.shape == (2000, 64, 64), X_train.shape
assert X_test.shape == (1036, 64, 64), X_test.shape
assert {"id", "log10_mass"}.issubset(labels.columns)
assert labels["id"].is_unique and set(labels["id"]) == set(range(2000))
# Dans le code des premiers essais, la cible était NumPy float64, conserver cette précision.
# Réordonner les masses par id évite un décalage accidentel image/cible.
y_train = labels.set_index("id").reindex(np.arange(2000))["log10_mass"].to_numpy()
assert y_train.shape == (2000,) and np.isfinite(y_train).all()
assert np.isfinite(X_train).all() and np.isfinite(X_test).all()

print("Images train/test :", X_train.shape, X_test.shape)
print("Masses : min %.5f, médiane %.5f, moyenne %.5f, max %.5f" %
      (y_train.min(), np.median(y_train), y_train.mean(), y_train.max()))

Images train/test : (2000, 64, 64) (1036, 64, 64)
Masses : min 11.26865, médiane 11.60403, moyenne 11.58758, max 11.84448


In [3]:
# 03. Exploration reproductible du décalage train/test (sans étiquettes test)

# Écart-type dans les coins : approximation du bruit de fond sans étiquettes.
def corner_noise_std(img, corner=6):
    corners = [img[:corner, :corner], img[:corner, -corner:],
               img[-corner:, :corner], img[-corner:, -corner:]]
    return float(np.std(np.concatenate([c.ravel() for c in corners])))

stats = pd.DataFrame({
    "train": [float(np.min(X_train)), float(np.max(X_train)),
              float(np.mean(X_train)), float(np.mean(X_train == 0)),
              float(np.mean(X_train < 0)),
              float(np.median([corner_noise_std(x) for x in X_train]))],
    "test": [float(np.min(X_test)), float(np.max(X_test)),
             float(np.mean(X_test)), float(np.mean(X_test == 0)),
             float(np.mean(X_test < 0)),
             float(np.median([corner_noise_std(x) for x in X_test]))]
}, index=["minimum", "maximum", "moyenne_pixel", "fraction_zeros",
          "fraction_negatifs", "sigma_fond_median"])
print(stats.round(6).to_string())
print("Bruit de simulation fixé :", NOISE_SIGMA)
# Pour mémoire : split 80/20 + CV5 employés pendant le développement.
# Stratification en 10 groupes de masses pour équilibrer les plis.
all_idx = np.arange(len(y_train))
all_bins = pd.qcut(y_train, q=10, labels=False, duplicates="drop")
idx_dev, idx_holdout = train_test_split(all_idx, test_size=0.20,
                                       random_state=SEED, stratify=all_bins)
dev_bins = pd.qcut(y_train[idx_dev], q=10, labels=False, duplicates="drop")
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
cv_splits = [(idx_dev[tr], idx_dev[va]) for tr,va in skf.split(idx_dev, dev_bins)]
assert len(idx_dev) == 1600 and len(idx_holdout) == 400
print("Développement :", len(idx_dev), "| Holdout historique :", len(idx_holdout))
# Le modèle FINAL sera réentraîné sur 2000 images (autorisé pour prédire test).

                       train       test
minimum             0.000000  -0.736223
maximum            39.867065  45.431805
moyenne_pixel       0.044243   0.043334
fraction_zeros      0.569336   0.000000
fraction_negatifs   0.000000   0.476068
sigma_fond_median   0.000000   0.133418
Bruit de simulation fixé : 0.133942
Développement : 1600 | Holdout historique : 400


## A. Prétraitement / caractéristiques physiques
La transformation géométrique évite de fixer arbitrairement le centre de l'arc au centre de l'image. Les **12 anneaux** et les **18 mesures de géométrie** sont extraits de l'image bruitée ; les **14 mesures complémentaires** proviennent d'un léger filtrage gaussien. Ces fonctions ont été reprises du notebook d'expérimentation déjà utilisé, plutôt que redéfinies arbitrairement.

In [4]:
# 04. Translation + caractéristiques radiales (24) et géométriques (18)
def shift_image(image, dy, dx):
    """Translate sans enroulement ; pixels hors cadre remplacés par zéro."""
    h, w = image.shape
    shifted = np.zeros_like(image)
    y_src0, y_src1 = max(0, -dy), min(h, h - dy)
    x_src0, x_src1 = max(0, -dx), min(w, w - dx)
    if y_src0 < y_src1 and x_src0 < x_src1:
        shifted[y_src0 + dy:y_src1 + dy, x_src0 + dx:x_src1 + dx] = (
            image[y_src0:y_src1, x_src0:x_src1]
        )
    return shifted

YY, XX = np.indices((64, 64), dtype=np.float64)

# Profils radiaux autour du centroïde de lumière, pas forcément du centre 32x32.
# Les 12 anneaux contiennent deux statistiques chacun : énergie et nombre de pixels.
def extract_radial24_single(image):
    image = np.asarray(image, dtype=np.float64)
    weights = np.maximum(image - 3 * NOISE_SIGMA, 0.0)
    total = weights.sum()
    if total > 0:
        cy = (YY * weights).sum() / total
        cx = (XX * weights).sum() / total
    else:
        cy = cx = 31.5
    r = np.hypot(YY - cy, XX - cx)
    rings = np.minimum((r / (32.0 / 12.0)).astype(int), 11)
    positive = np.maximum(image - 2 * NOISE_SIGMA, 0.0)
    significant = (image > 3 * NOISE_SIGMA).astype(np.float64)
    intensity = np.bincount(rings.ravel(), weights=positive.ravel(), minlength=12)
    counts = np.bincount(rings.ravel(), weights=significant.ravel(), minlength=12)
    return np.stack([np.log1p(intensity), np.log1p(counts)], axis=1).ravel().astype(np.float32)

# Quantile radial pondéré : rayon sous lequel se trouve une fraction q de la lumière.
def weighted_radius_quantile(radius, signal, q):
    r = radius.ravel()
    w = signal.ravel()
    order = np.argsort(r)
    cdf = np.cumsum(w[order])
    if cdf.size == 0 or cdf[-1] <= 0:
        return 0.0
    pos = min(np.searchsorted(cdf, q * cdf[-1]), len(order) - 1)
    return float(r[order[pos]])

# 18 mesures : rayons, excentricité, harmoniques angulaires, ajustement de cercle.
def geometry_features(image):
    image = np.asarray(image, dtype=np.float64)
    signal = np.maximum(image - 3 * NOISE_SIGMA, 0.0)
    total = float(signal.sum())
    if total < 1.0:
        return np.zeros(18, dtype=np.float32)

    cy = float((YY * signal).sum() / total)
    cx = float((XX * signal).sum() / total)
    dy, dx = YY - cy, XX - cx
    r = np.hypot(dx, dy)
    mean_r = float((signal * r).sum() / total)
    std_r = float(np.sqrt((signal * (r - mean_r)**2).sum() / total))
    r20 = weighted_radius_quantile(r, signal, .20)
    r50 = weighted_radius_quantile(r, signal, .50)
    r80 = weighted_radius_quantile(r, signal, .80)

    var_x = float((signal * dx**2).sum() / total)
    var_y = float((signal * dy**2).sum() / total)
    cov_xy = float((signal * dx * dy).sum() / total)
    eig = np.maximum(np.linalg.eigvalsh([[var_x, cov_xy], [cov_xy, var_y]]), 0)
    minor, major = float(np.sqrt(eig[0])), float(np.sqrt(eig[1]))
    elongation = min(major / (minor + 1e-6), 100.0)
    theta = np.arctan2(dy, dx)
    amp2 = float(abs((signal * np.exp(2j * theta)).sum() / total))
    amp4 = float(abs((signal * np.exp(4j * theta)).sum() / total))
    frac8 = float(signal[r >= 8].sum() / total)
    frac16 = float(signal[r >= 16].sum() / total)

    # Ajustement algébrique d'un cercle aux pixels significatifs.
    # Un cercle plausible peut approximer la structure des arcs de lentille.
    circle_radius = circle_residual = circle_offset = valid_circle = 0.0
    threshold = max(0.10 * float(signal.max()), NOISE_SIGMA)
    mask = signal > threshold
    if np.count_nonzero(mask) >= 8:
        xs, ys = XX[mask], YY[mask]
        mat = np.column_stack((2 * xs, 2 * ys, np.ones(len(xs))))
        rhs = xs**2 + ys**2
        try:
            fx, fy, c = np.linalg.lstsq(mat, rhs, rcond=None)[0]
            radius_sq = c + fx**2 + fy**2
            fitted_radius = np.sqrt(max(radius_sq, 0.0))
            offset = np.hypot(fx - cx, fy - cy)
            if 1.0 < fitted_radius < 50.0 and offset < 50.0:
                circle_radius = float(fitted_radius)
                circle_offset = float(offset)
                circle_residual = float(np.mean(
                    np.abs(np.hypot(xs - fx, ys - fy) - fitted_radius)
                ))
                valid_circle = 1.0
        except np.linalg.LinAlgError:
            pass

    out = np.array([
        np.log1p(total), np.log1p(np.count_nonzero(signal)),
        mean_r, std_r, r20, r50, r80,
        major, minor, elongation, amp2, amp4, frac8, frac16,
        circle_radius, circle_residual, circle_offset, valid_circle
    ], dtype=np.float32)
    assert out.shape == (18,)
    return out

# Retourne les 24 features radiales puis les 42 features concaténées.
def extract_both_features(images):
    radial24 = np.stack([extract_radial24_single(img) for img in images])
    geom18 = np.stack([geometry_features(img) for img in images])
    extended42 = np.hstack([radial24, geom18]).astype(np.float32)
    return radial24, extended42

In [5]:
# 05. 14 caractéristiques après lissage (sigma spatial 1.0)
# On estime ici le bruit résiduel après un filtrage gaussien sigma = 1 pixel.
# Le seuil est abaissé : on peut alors étudier les arcs plus faibles.
SMOOTH_SIGMA = 1.0
impulse = np.zeros((17, 17), dtype=np.float64)
impulse[8, 8] = 1.0
kernel = gaussian_filter(impulse, sigma=SMOOTH_SIGMA, mode="constant")
SIGMA_SMOOTHED = float(NOISE_SIGMA * np.sqrt(np.sum(kernel**2)))

# Caractéristiques de flux, de distribution radiale, de forme et de symétrie.
def extract_smoothed14_single(image):
    smooth = gaussian_filter(np.asarray(image, dtype=np.float64), sigma=SMOOTH_SIGMA, mode="reflect")
    signal = np.maximum(smooth - 2.5 * SIGMA_SMOOTHED, 0.0)
    total = float(signal.sum())
    if total < 1e-8:
        return np.zeros(14, dtype=np.float32)

    cy = float((YY * signal).sum() / total)
    cx = float((XX * signal).sum() / total)
    dy, dx = YY - cy, XX - cx
    r = np.hypot(dx, dy)
    mean_r = float((signal * r).sum() / total)
    std_r = float(np.sqrt((signal * (r - mean_r)**2).sum() / total))
    radial_bins = np.clip(r.astype(int), 0, 63)
    hist = np.bincount(radial_bins.ravel(), weights=signal.ravel(), minlength=64)
    cumulative = np.cumsum(hist)
    r50 = int(np.searchsorted(cumulative, 0.5 * total))
    r90 = int(np.searchsorted(cumulative, 0.9 * total))

    var_x = float((signal * dx**2).sum() / total)
    var_y = float((signal * dy**2).sum() / total)
    cov_xy = float((signal * dx * dy).sum() / total)
    eig = np.maximum(np.linalg.eigvalsh([[var_x, cov_xy], [cov_xy, var_y]]), 0)
    minor, major = float(np.sqrt(eig[0])), float(np.sqrt(eig[1]))
    elongation = min(major / (minor + 1e-6), 100.0)
    theta = np.arctan2(dy, dx)
    amp2 = float(abs((signal * np.exp(2j * theta)).sum() / total))
    amp4 = float(abs((signal * np.exp(4j * theta)).sum() / total))

    return np.array([
        np.log1p(total), np.log1p(np.count_nonzero(smooth > 3 * SIGMA_SMOOTHED)),
        np.log1p(signal.max()), mean_r, std_r, r50, r90,
        major, minor, elongation, amp2, amp4,
        float(signal[r >= 16].sum() / total),
        float(signal[r >= 24].sum() / total)
    ], dtype=np.float32)

def extract_smoothed14_batch(images):
    return np.stack([extract_smoothed14_single(img) for img in images])

# Base V8 : 42 features brutes + 14 features après lissage.
def extract_features56(images):
    _, f42 = extract_both_features(images)
    f14 = extract_smoothed14_batch(images)
    return np.hstack([f42, f14]).astype(np.float32)

## B. Débruiteur U-Net **entraîné depuis zéro**
Le U-Net ne vient d'aucun modèle externe. L'entrée est une image de la compétition translatée et contaminée par un bruit gaussien indépendant ; la cible est **la même image sans le bruit**. Les intensités sont transformées par `arcsinh(x / 0.2)`. Les poids apprennent la reconstruction ; la branche de régression utilisera ensuite 14 mesures sur l'image reconstruite. Les données de test ne servent jamais d'exemples étiquetés.

In [6]:
# 06. Dataset de débruitage et architecture U-Net (370 689 paramètres)
# Les paires (image bruitée, image propre) sont construites à partir de nos
# seules images d'entraînement ; elles utilisent la même translation cible/entrée.
class DenoisingGalaxyDataset(Dataset):
    def __init__(self, images, training=True, seed=2026):
        self.images = np.asarray(images, dtype=np.float32)
        self.training = training
        self.seed = int(seed)
        self.rng = np.random.default_rng(seed)
    def __len__(self): return len(self.images)
    def __getitem__(self, idx):
        clean = self.images[idx].copy()
        rng = self.rng if self.training else np.random.default_rng(self.seed + int(idx))
        dy, dx = rng.integers(-10, 11, size=2)
        clean = shift_image(clean, int(dy), int(dx))
        noise = rng.normal(0, NOISE_SIGMA, clean.shape).astype(np.float32)
        noisy = clean + noise
        noisy = np.arcsinh(noisy / INTENSITY_SCALE).astype(np.float32)
        clean = np.arcsinh(clean / INTENSITY_SCALE).astype(np.float32)
        return torch.from_numpy(noisy[None]), torch.from_numpy(clean[None])

class DenoiseBlock(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.layers = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, 3, padding=1),
            nn.GroupNorm(8, out_channels), nn.SiLU(),
            nn.Conv2d(out_channels, out_channels, 3, padding=1),
            nn.GroupNorm(8, out_channels), nn.SiLU())
    def forward(self,x): return self.layers(x)

# U-Net résiduel : reconstruit une correction d'image, non la masse directement.
# 370689 paramètres, initialisés aléatoirement au démarrage de l'apprentissage.
class GalaxyDenoiser(nn.Module):
    def __init__(self):
        super().__init__()
        self.enc1 = DenoiseBlock(1, 32)
        self.enc2 = DenoiseBlock(32, 64)
        self.bottleneck = DenoiseBlock(64, 96)
        self.dec2 = DenoiseBlock(96+64, 64)
        self.dec1 = DenoiseBlock(64+32, 32)
        self.out = nn.Conv2d(32, 1, kernel_size=1)
        nn.init.zeros_(self.out.weight)
        nn.init.zeros_(self.out.bias)
    def forward(self, x):
        e1 = self.enc1(x)
        e2 = self.enc2(F.max_pool2d(e1,2))
        b = self.bottleneck(F.max_pool2d(e2,2))
        u2 = F.interpolate(b, size=e2.shape[-2:], mode="bilinear", align_corners=False)
        d2 = self.dec2(torch.cat([u2,e2],dim=1))
        u1 = F.interpolate(d2, size=e1.shape[-2:], mode="bilinear", align_corners=False)
        d1 = self.dec1(torch.cat([u1,e1],dim=1))
        return x + self.out(d1)

# Pondération : donner plus d'importance aux pixels portant de la lumière.
def weighted_reconstruction_loss(pred, clean):
    weights = 1.0 + 2.0*torch.clamp(torch.abs(clean), min=0.0, max=1.0)
    return (weights*torch.abs(pred-clean)).sum()/weights.sum()

# Parcours d'une époque du débruiteur (perte moyenne sur les exemples).
def train_denoiser_epoch(model, loader, optimizer, dev=DEVICE):
    model.train()
    total, n = 0., 0
    for noisy, clean in loader:
        noisy = noisy.to(dev, non_blocking=True)
        clean = clean.to(dev, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)
        reconstructed = model(noisy)
        loss = weighted_reconstruction_loss(reconstructed, clean)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        total += loss.item() * noisy.size(0)
        n += noisy.size(0)
    return total/n

@torch.inference_mode()
def evaluate_denoiser(model, loader, dev=DEVICE):
    model.eval()
    total, n = 0., 0
    for noisy, clean in loader:
        noisy, clean = noisy.to(dev), clean.to(dev)
        loss = weighted_reconstruction_loss(model(noisy), clean)
        total += loss.item()*noisy.size(0)
        n += noisy.size(0)
    return total/n

print("Paramètres U-Net :", sum(p.numel() for p in GalaxyDenoiser().parameters()))
assert sum(p.numel() for p in GalaxyDenoiser().parameters()) == 370689

Paramètres U-Net : 370689


In [7]:
# 07 — Charger le U-Net ORIGINAL s'il est lisible, sinon l'entraîner depuis zéro.
# Les poids de référence ne sont JAMAIS écrasés par une régénération.
DENOISER_PATH = RESULTS_DIR / "denoiser_full_v10.pt"
DENOISER_REBUILT_PATH = RESULTS_DIR / "denoiser_full_v10_regenere.pt"


def train_final_denoiser(save_path):
    """Entraîner le U-Net 65 époques sur les 2000 images autorisées."""
    seed = AUG_SEED
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

    # A chaque appel au Dataset, une nouvelle translation et un nouveau bruit sont tirés.
    dataset = DenoisingGalaxyDataset(X_train, training=True, seed=seed)
    loader = DataLoader(
        dataset, batch_size=32, shuffle=True, num_workers=0,
        pin_memory=torch.cuda.is_available(),
        generator=torch.Generator().manual_seed(seed)
    )
    model = GalaxyDenoiser().to(DEVICE)
    optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=1e-5)
    history = []

    for epoch in range(1, 66):
        loss = train_denoiser_epoch(model, loader, optimizer)
        history.append((epoch, loss))
        if epoch == 1 or epoch % 5 == 0:
            print(f"Époque {epoch:02d}/65 | perte de débruitage {loss:.6f}")

    pd.DataFrame(history, columns=["epoch", "weighted_train_mae"]).to_csv(
        RESULTS_DIR / "denoiser_full_v10_history_regenere.csv", index=False
    )
    torch.save({
        "model_state_dict": {
            key: value.detach().cpu().clone() for key, value in model.state_dict().items()
        },
        "epochs": 65, "seed": seed, "noise_sigma": NOISE_SIGMA,
        "intensity_scale": INTENSITY_SCALE, "n_training_images": 2000
    }, save_path)
    model.eval()
    print("U-Net réentraîné et sauvegardé :", save_path)
    return model


def try_load_denoiser(path):
    """Charger un checkpoint local de confiance et vérifier sa compatibilité."""
    try:
        checkpoint = torch.load(path, map_location="cpu", weights_only=True)
        assert np.isclose(checkpoint["noise_sigma"], NOISE_SIGMA)
        assert np.isclose(checkpoint["intensity_scale"], INTENSITY_SCALE)
        model = GalaxyDenoiser().to(DEVICE)
        model.load_state_dict(checkpoint["model_state_dict"], strict=True)
        model.eval()
        return model
    except Exception as error:
        print(f"ATTENTION : U-Net illisible ({path}) : {type(error).__name__}: {error}")
        return None

# Seul le checkpoint ORIGINAL permet de charger tel quel le Boosting70 original.
DENOISER_ORIGINAL_LOADED = False
denoiser_final = None
if USE_SAVED_MODELS:
    if DENOISER_PATH.is_file():
        denoiser_final = try_load_denoiser(DENOISER_PATH)
        if denoiser_final is not None:
            DENOISER_ORIGINAL_LOADED = True
            print("U-Net ORIGINAL chargé :", DENOISER_PATH)
    if denoiser_final is None and DENOISER_REBUILT_PATH.is_file():
        denoiser_final = try_load_denoiser(DENOISER_REBUILT_PATH)
        if denoiser_final is not None:
            print("U-Net reconstruit précédemment chargé :", DENOISER_REBUILT_PATH)

if denoiser_final is None:
    # Ne pas écraser le checkpoint ORIGINAL : sauvegarde à un autre nom.
    print("U-Net indisponible (ou réentraînement forcé) : reconstruction sur 2000 images...")
    denoiser_final = train_final_denoiser(DENOISER_REBUILT_PATH)

print("Checkpoint U-Net original utilisé :", DENOISER_ORIGINAL_LOADED)


U-Net indisponible (ou réentraînement forcé) : reconstruction sur 2000 images...
Époque 01/65 | perte de débruitage 0.342965
Époque 05/65 | perte de débruitage 0.033706
Époque 10/65 | perte de débruitage 0.029704
Époque 15/65 | perte de débruitage 0.027497
Époque 20/65 | perte de débruitage 0.027227
Époque 25/65 | perte de débruitage 0.025821
Époque 30/65 | perte de débruitage 0.024988
Époque 35/65 | perte de débruitage 0.025144
Époque 40/65 | perte de débruitage 0.024584
Époque 45/65 | perte de débruitage 0.023903
Époque 50/65 | perte de débruitage 0.024061
Époque 55/65 | perte de débruitage 0.025371
Époque 60/65 | perte de débruitage 0.023363
Époque 65/65 | perte de débruitage 0.023172
U-Net réentraîné et sauvegardé : results\denoiser_full_v10_regenere.pt
Checkpoint U-Net original utilisé : False


In [8]:
# 08. Reconstruction des images et 14 mesures géométriques débruitées
# L'inférence du débruiteur ne doit pas calculer de gradients.
# Les images Kaggle sont déjà bruitées : aucun bruit supplémentaire n'est ajouté.
@torch.inference_mode()
def reconstruct_images(images, batch_size=64, denoiser=None):
    if denoiser is None: denoiser=denoiser_final
    denoiser.eval()
    images=np.asarray(images,dtype=np.float32)
    parts=[]
    for start in range(0,len(images),batch_size):
        batch=images[start:start+batch_size]
        transformed=np.arcsinh(batch/INTENSITY_SCALE).astype(np.float32)
        t=torch.from_numpy(transformed[:,None]).to(DEVICE)
        reconstructed=denoiser(t)
        intens=INTENSITY_SCALE*torch.sinh(torch.clamp(reconstructed,-10,10))
        intens=torch.clamp(intens,min=0)
        parts.append(intens[:,0].cpu().numpy())
    return np.concatenate(parts,axis=0)

# Seuil de résidu après débruitage, fixé pendant les essais V10.
DENOISED_THRESHOLD=0.02

# Mesures géométriques tirées des images restaurées (14 variables).
def denoised_features14(image):
    image=np.asarray(image,dtype=np.float64)
    signal=np.maximum(image-DENOISED_THRESHOLD,0)
    total=float(signal.sum())
    if total<1e-8: return np.zeros(14,dtype=np.float32)
    cy=float((YY*signal).sum()/total)
    cx=float((XX*signal).sum()/total)
    dy,dx=YY-cy,XX-cx
    r=np.hypot(dx,dy)
    mean_r=(signal*r).sum()/total
    std_r=np.sqrt((signal*(r-mean_r)**2).sum()/total)
    radial_bins=np.clip(r.astype(np.int32),0,63)
    radial_hist=np.bincount(radial_bins.ravel(),weights=signal.ravel(),minlength=64)
    cumulative=np.cumsum(radial_hist)
    r50=np.searchsorted(cumulative, total*.5)
    r90=np.searchsorted(cumulative, total*.9)
    var_x=(signal*dx**2).sum()/total
    var_y=(signal*dy**2).sum()/total
    cov_xy=(signal*dx*dy).sum()/total
    eig=np.maximum(np.linalg.eigvalsh(np.array([[var_x,cov_xy],[cov_xy,var_y]])),0)
    minor,major=np.sqrt(eig[0]),np.sqrt(eig[1])
    elongation=min(major/(minor+1e-6),100.)
    theta=np.arctan2(dy,dx)
    def angular_amplitude(order):
        c=(signal*np.cos(order*theta)).sum()/total
        s=(signal*np.sin(order*theta)).sum()/total
        return np.hypot(c,s)
    result=np.asarray([
        np.log1p(total),np.log1p(np.count_nonzero(signal)),np.log1p(image.max()),
        mean_r,std_r,r50,r90,major,minor,elongation,
        angular_amplitude(2),angular_amplitude(4),
        signal[r>=16].sum()/total,signal[r>=24].sum()/total
    ],dtype=np.float32)
    assert result.shape==(14,)
    return result

def extract_denoised14_batch(images, denoiser=None):
    reconstructed=reconstruct_images(images,denoiser=denoiser)
    return np.stack([denoised_features14(im) for im in reconstructed])

# Architecture finale : 42 brut + 14 lissé + 14 U-Net = 70.
def extract_v10_features(images, denoiser=None):
    _, f42=extract_both_features(images)
    f14_smooth=extract_smoothed14_batch(images)
    f14_unet=extract_denoised14_batch(images,denoiser=denoiser)
    f70=np.hstack([f42,f14_smooth,f14_unet]).astype(np.float32)
    assert f42.shape==(len(images),42) and f70.shape==(len(images),70)
    assert np.isfinite(f70).all()
    return f42,f70

# Test de propagation sur deux images (aucune prédiction de masse ici)
_probe=np.random.default_rng(7).normal(0,NOISE_SIGMA,(2,64,64)).astype(np.float32)
f42_probe,f70_probe=extract_v10_features(X_train[:2]+_probe)
print("Caractéristiques 42/70 :",f42_probe.shape,f70_probe.shape)

Caractéristiques 42/70 : (2, 42) (2, 70)


## C. Régression V10 : Boosting70 + ExtraTrees42

Le **Boosting70** utilise les informations provenant du U-Net final : il faut donc le charger uniquement avec **le même débruiteur**. Extra Trees utilise les 42 variables initiales et n'a pas besoin du U-Net.

### Piste de restauration fidèle

V8 était composé de `0,70 × Boosting56 + 0,30 × ExtraTrees42` ; V10 était composé de `0,70 × Boosting70 + 0,30 × ExtraTrees42`. Si les mêmes caractéristiques et checkpoints originaux sont présents, on peut donc isoler **la contribution du même Extra Trees sur le jeu de test** à partir du CSV V8, puis l'ajouter au Boosting70 original.

Le CSV V10 historique sert à **contrôler** cette reconstruction. Nous ne le lisons pas pour produire les prédictions.

Si la restauration échoue, le notebook fournit encore la procédure complète de réentraînement à partir des 2 000 images, mais ne garantit pas une MAE identique. Pour retrouver intégralement le modèle Extra Trees d'origine, la meilleure solution reste de récupérer un `.joblib` lisible dans l'environnement où il a été sauvegardé.


### Trois scénarios possibles pour `extra_trees_v7.joblib`

| Situation | Action du notebook | Reproductibilité |
|---|---|---|
| `.joblib` original lisible | Calcul des prédictions avec le modèle original | Audit numérique avec V10 |
| `.joblib` illisible, CSV V8/V7 et Boosting historique disponibles | Calcul **algébrique des prédictions Extra Trees sur le test uniquement** | Peut retrouver le CSV original ; pas un modèle réentraîné |
| Aucun artefact historique utilisable | Réentraînement du modèle Extra Trees depuis les 2 000 images | Modèle valide mais **différent** ; CSV `alternative` si audit non conforme |

Les prédictions `id,log10_mass` historiques sont des **sorties de nos modèles**, et non des étiquettes de vérité terrain. Aucun score Kaggle n'est utilisé pour entraîner un régresseur.


In [9]:
# 09 — Récupération des modèles : ne PAS réentraîner silencieusement Extra Trees.
# ------------------------------------------------------------------------------
# 1. Le Boosting70 d'origine est chargé seulement si le U-Net d'origine est chargé.
# 2. ExtraTrees42 d'origine est essayé séparément. Si le pickle est illisible,
#    son absence est conservée : la cellule suivante tentera de récupérer ses
#    SORTIES DE TEST historiques depuis V8/V7, avant d'envisager un réentraînement.
# 3. Tous les modèles reconstitués sont sauvegardés sous un nom distinct.

BOOST_PATH = RESULTS_DIR / 'boosting_v10_unet_geometry.joblib'
EXTRA_PATH = RESULTS_DIR / 'extra_trees_v7.joblib'
BOOST_REBUILT_PATH = RESULTS_DIR / 'boosting_v10_unet_geometry_regenere.joblib'
EXTRA_REBUILT_PATH = RESULTS_DIR / 'extra_trees_v7_regenere.joblib'
BOOST_REBUILT_META_PATH = RESULTS_DIR / 'boosting_v10_unet_geometry_regenere.meta.json'


def unet_weights_sha256(model):
    """Empreinte des poids, pour éviter de mélanger un Boosting avec un autre U-Net."""
    digest = hashlib.sha256()
    for name, tensor in sorted(model.state_dict().items()):
        digest.update(name.encode('utf-8'))
        digest.update(tensor.detach().cpu().contiguous().numpy().tobytes())
    return digest.hexdigest()


CURRENT_UNET_HASH = unet_weights_sha256(denoiser_final)


def usable_rebuilt_boosting():
    """Un Boosting reconstruit n'est valide qu'avec le même U-Net reconstruit."""
    if not (BOOST_REBUILT_META_PATH.exists() and BOOST_REBUILT_PATH.exists()):
        return False
    try:
        metadata = json.loads(BOOST_REBUILT_META_PATH.read_text(encoding='utf-8'))
        return metadata.get('unet_weights_sha256') == CURRENT_UNET_HASH
    except (OSError, json.JSONDecodeError):
        return False


def iter_final_augmented_images():
    """Répète les copies et tirages utilisés lors de la soumission V10."""
    for copy_id in range(N_COPIES):
        noise_rng = np.random.default_rng(AUG_SEED + 100 + copy_id)
        noise = noise_rng.normal(0, NOISE_SIGMA, X_train.shape).astype(np.float32)
        if copy_id == 0:
            shifted = X_train
        else:
            shift_rng = np.random.default_rng(AUG_SEED + 200 + copy_id)
            shifts = shift_rng.integers(-MAX_SHIFT, MAX_SHIFT + 1, size=(len(X_train), 2))
            shifted = np.stack([
                shift_image(img, int(dy), int(dx))
                for img, (dy, dx) in zip(X_train, shifts)
            ])
        yield copy_id, shifted + noise


def final_augmented_batches(rebuild_boosting=True, rebuild_extra=True):
    """Caractéristiques supervisées sur 2 000 images, avec cinq augmentations."""
    f42_batches, f70_batches = [], []
    for copy_id, augmented in iter_final_augmented_images():
        if rebuild_boosting:
            f42, f70 = extract_v10_features(augmented)
            f70_batches.append(f70)
            if rebuild_extra:
                f42_batches.append(f42)
        elif rebuild_extra:
            _, f42 = extract_both_features(augmented)
            f42_batches.append(f42)
        print(f'Augmentation {copy_id + 1}/{N_COPIES} extraite')
    f42all = np.vstack(f42_batches) if rebuild_extra else None
    f70all = np.vstack(f70_batches) if rebuild_boosting else None
    y_aug = np.tile(y_train, N_COPIES)
    if rebuild_boosting:
        assert f70all.shape == (10000, 70) and np.isfinite(f70all).all()
    if rebuild_extra:
        assert f42all.shape == (10000, 42) and np.isfinite(f42all).all()
    return f42all, f70all, y_aug


def try_load_regressor(path, expected_type, feature_count, label):
    """Lire uniquement nos checkpoints de confiance, puis vérifier leur schéma."""
    if not path.exists():
        print(f'{label} absent : {path}')
        return None
    try:
        model = joblib.load(path)
        if not isinstance(model, expected_type):
            raise TypeError(f'{label}: classe inattendue {type(model).__name__}')
        if getattr(model, 'n_features_in_', None) != feature_count:
            raise ValueError(f'{label}: {model.n_features_in_} variables au lieu de {feature_count}')
        print(f'{label} chargé : {path} ({path.stat().st_size:,} octets)')
        return model
    except Exception as exc:
        print(f'ATTENTION : échec {label} : {type(exc).__name__} : {exc}')
        print('Le fichier original est laissé intact.')
        return None


# Boosting final : charger le checkpoint de V10, sinon réentraîner uniquement celui-ci.
boosting_v10 = None
boost_source = None
if USE_SAVED_MODELS and DENOISER_ORIGINAL_LOADED:
    boosting_v10 = try_load_regressor(
        BOOST_PATH, HistGradientBoostingRegressor, 70, 'Boosting70 original'
    )
    if boosting_v10 is not None:
        boost_source = str(BOOST_PATH)

if USE_SAVED_MODELS and boosting_v10 is None and usable_rebuilt_boosting():
    boosting_v10 = try_load_regressor(
        BOOST_REBUILT_PATH, HistGradientBoostingRegressor, 70, 'Boosting70 régénéré'
    )
    if boosting_v10 is not None:
        boost_source = str(BOOST_REBUILT_PATH)

rebuild_boosting = boosting_v10 is None
if rebuild_boosting:
    print('Boosting70 absent : réentraînement sur 10 000 exemples augmentés…')
    _, f70all, y_aug = final_augmented_batches(rebuild_boosting=True, rebuild_extra=False)
    boosting_v10 = HistGradientBoostingRegressor(**HGB_PARAMS).fit(f70all, y_aug)
    joblib.dump(boosting_v10, BOOST_REBUILT_PATH)
    BOOST_REBUILT_META_PATH.write_text(
        json.dumps({'unet_weights_sha256': CURRENT_UNET_HASH}, indent=2), encoding='utf-8'
    )
    boost_source = str(BOOST_REBUILT_PATH)

# Extra Trees : seul le checkpoint ORIGINAL est chargé ici.
# Le modèle régénéré (0.02905 sur Kaggle en combinaison) n'est pas substitué
# sans contrôle : la cellule 10 utilisera d'abord l'archive de prédictions V8/V7.
extra_v10 = None
if USE_SAVED_MODELS:
    extra_v10 = try_load_regressor(
        EXTRA_PATH, ExtraTreesRegressor, 42, 'ExtraTrees42 original'
    )

REGRESSOR_SOURCES = {
    'boosting': boost_source,
    'extra_trees': str(EXTRA_PATH) if extra_v10 is not None else None,
}
RETRAINED_COMPONENTS = {
    'denoiser': not DENOISER_ORIGINAL_LOADED,
    'boosting': rebuild_boosting,
    'extra_trees': False,
}
print('\n=== MODÈLES : CHARGEMENT INITIAL ===')
print(REGRESSOR_SOURCES)
print('ExtraTrees original disponible :', extra_v10 is not None)


Boosting70 absent : réentraînement sur 10 000 exemples augmentés…
Augmentation 1/5 extraite
Augmentation 2/5 extraite
Augmentation 3/5 extraite
Augmentation 4/5 extraite
Augmentation 5/5 extraite
ExtraTrees42 original absent : results\extra_trees_v7.joblib

=== MODÈLES : CHARGEMENT INITIAL ===
{'boosting': 'results\\boosting_v10_unet_geometry_regenere.joblib', 'extra_trees': None}
ExtraTrees original disponible : False


In [10]:
# 10 — Inférence V10 : reconstituer la composante Extra Trees avant tout re-fit.
# ------------------------------------------------------------------------------
# L'archive V8 contient les 1 036 prédictions de 0.70*B56 + 0.30*E42.
# Comme V10 réutilise E42, on peut isoler E42_test = (V8 - 0.70*B56)/0.30.
# Cela retrouve les PRÉDICTIONS de ce modèle, PAS son état entraîné ni sa capacité
# à prédire de nouvelles images. Cette distinction figure dans le README.


def read_historical_csv(path):
    """Vérifier le format et rétablir l'ordre exact des images .npy (id 0..1035)."""
    frame = pd.read_csv(path)
    assert {'id', 'log10_mass'}.issubset(frame.columns)
    assert len(frame) == len(X_test) and frame['id'].is_unique
    frame = frame.sort_values('id').reset_index(drop=True)
    assert np.array_equal(frame['id'].to_numpy(), np.arange(len(X_test)))
    assert np.isfinite(frame['log10_mass']).all()
    return frame


def retrieve_extra_from_archives(f42test, f70test):
    """Récupérer les sorties test de l'Extra Trees historique via V8 ou V7.

    Les deux voies sont indépendantes et sont comparées si toutes deux sont disponibles.
    Aucune valeur y_test ni aucun CSV V10 original n'est consulté ici.
    """
    if not (USE_SAVED_MODELS and DENOISER_ORIGINAL_LOADED and boost_source == str(BOOST_PATH)):
        print('Récupération historique désactivée : modèles originaux V10 non disponibles.')
        return None, None

    archives = [
        ('V8', SUBMISSIONS_DIR / 'submission_v8_smoothed_blend.csv',
         RESULTS_DIR / 'boosting_v8_smoothed.joblib', 56, f70test[:, :56]),
        ('V7', SUBMISSIONS_DIR / 'submission_v7_boosting_extratrees.csv',
         RESULTS_DIR / 'boosting_v4_geometry.joblib', 42, f42test),
    ]
    recovered = []
    for name, csv_path, booster_path, count, features in archives:
        if not (csv_path.exists() and booster_path.exists()):
            print(f'Archive {name} absente ou incomplète : {csv_path.name} / {booster_path.name}')
            continue
        try:
            old_boost = try_load_regressor(
                booster_path, HistGradientBoostingRegressor, count, f'Boosting {name}'
            )
            if old_boost is None:
                continue
            old_csv = read_historical_csv(csv_path)
            old_boost_predictions = old_boost.predict(features)
            extra_predictions = (
                old_csv['log10_mass'].to_numpy() - .70 * old_boost_predictions
            ) / .30
            if not np.isfinite(extra_predictions).all():
                raise ValueError('Prédictions Extra Trees non finies')
            recovered.append((name, extra_predictions))
            print(f'Contribution ExtraTrees historique récupérée via {name}.')
        except Exception as exc:
            print(f'Archive {name} non exploitable : {type(exc).__name__}: {exc}')

    if len(recovered) >= 2:
        cross = np.abs(recovered[0][1] - recovered[1][1])
        print('=== CONTRÔLE CROISÉ V8 / V7 (ExtraTrees) ===')
        print('Écart absolu moyen :', float(cross.mean()))
        print('Écart absolu maximal :', float(cross.max()))
        if cross.max() > 1e-5:
            warnings.warn(
                'V8 et V7 ne permettent pas de récupérer les mêmes sorties Extra Trees. '
                'Il peut exister une différence de preprocessing / de checkpoint.', RuntimeWarning
            )

    if recovered:
        return recovered[0][1], 'prédictions historiques récupérées via ' + recovered[0][0]
    return None, None


# Test réel : pas de nouvelle translation, pas de bruit ajouté.
F_test42, F_test70 = extract_v10_features(X_test)
assert F_test42.shape == (1036, 42) and F_test70.shape == (1036, 70)
pred_boost70 = boosting_v10.predict(F_test70)

# 1. Extra Trees original ; 2. sortie historique V8/V7 ; 3. modèle régénéré.
PREDICTION_MODE = None
if extra_v10 is not None:
    pred_extra42 = extra_v10.predict(F_test42)
    PREDICTION_MODE = 'checkpoint ExtraTrees original chargé'
else:
    pred_extra42, PREDICTION_MODE = retrieve_extra_from_archives(F_test42, F_test70)

if pred_extra42 is None:
    # Cette voie réalise une V10 réentraînée, pas nécessairement la V10 à 0.02808.
    print('\nArchives indisponibles : réentraînement Extra Trees sur les données autorisées.')
    extra_v10 = None
    if USE_SAVED_MODELS and EXTRA_REBUILT_PATH.is_file():
        extra_v10 = try_load_regressor(
            EXTRA_REBUILT_PATH, ExtraTreesRegressor, 42, 'ExtraTrees42 régénéré'
        )
    if extra_v10 is None:
        f42all, _, y_aug = final_augmented_batches(
            rebuild_boosting=False, rebuild_extra=True
        )
        extra_v10 = ExtraTreesRegressor(**EXTRA_PARAMS).fit(f42all, y_aug)
        joblib.dump(extra_v10, EXTRA_REBUILT_PATH)
        RETRAINED_COMPONENTS['extra_trees'] = True
    pred_extra42 = extra_v10.predict(F_test42)
    PREDICTION_MODE = 'ExtraTrees42 régénéré : non identique à priori'
    REGRESSOR_SOURCES['extra_trees'] = str(EXTRA_REBUILT_PATH)
else:
    REGRESSOR_SOURCES['extra_trees'] = PREDICTION_MODE

# Vérifier le mélange validé pendant les expériences.
pred_v10 = .70 * pred_boost70 + .30 * pred_extra42
assert pred_v10.shape == (1036,) and np.isfinite(pred_v10).all()

# IMPORTANT : on ne consulte le CSV original que maintenant, pour l'audit.
REFERENCE_PATH = SUBMISSIONS_DIR / 'submission_v10_unet_geometry.csv'
if REFERENCE_PATH.exists():
    original = read_historical_csv(REFERENCE_PATH)
    discrepancies = np.abs(pred_v10 - original['log10_mass'].to_numpy())
    reproduced = bool(np.all(discrepancies <= 1e-7))
    print('\n=== AUDIT : CSV V10 ORIGINAL ===')
    print('Différence moyenne absolue :', float(discrepancies.mean()))
    print('Différence maximale :', float(discrepancies.max()))
    print('Identique à 1e-7 :', reproduced)
else:
    reproduced = None
    print('\nCSV V10 original absent : fidélité numérique NON VÉRIFIABLE.')

# On n'appelle « reproduite » que la sortie qui passe réellement l'audit.
if reproduced is True:
    OUTPUT_PATH = SUBMISSIONS_DIR / 'submission_v10_reproduite.csv'
else:
    OUTPUT_PATH = SUBMISSIONS_DIR / 'submission_v10_alternative.csv'
    warnings.warn(
        'Reproduction stricte non démontrée ; le CSV est nommé « alternative ». '
        'Ne pas remplacer la soumission V10 historique à 0.02808.', RuntimeWarning
    )

submission = pd.DataFrame({'id': np.arange(1036), 'log10_mass': pred_v10})
submission.to_csv(OUTPUT_PATH, index=False, float_format='%.8f')
assert read_historical_csv(OUTPUT_PATH).shape == (1036, 2)

print('\n=== SOUMISSION RECONSTRUITE ===')
print('Mode :', PREDICTION_MODE)
print('Fichier :', OUTPUT_PATH.resolve())
print('Original numériquement reproduit :', reproduced)
print(submission.head(10).to_string(index=False))


Récupération historique désactivée : modèles originaux V10 non disponibles.

Archives indisponibles : réentraînement Extra Trees sur les données autorisées.
Augmentation 1/5 extraite
Augmentation 2/5 extraite
Augmentation 3/5 extraite
Augmentation 4/5 extraite
Augmentation 5/5 extraite

CSV V10 original absent : fidélité numérique NON VÉRIFIABLE.

=== SOUMISSION RECONSTRUITE ===
Mode : ExtraTrees42 régénéré : non identique à priori
Fichier : C:\Users\yassi\Desktop\IFT_COMP2\submissions\submission_v10_alternative.csv
Original numériquement reproduit : None
 id  log10_mass
  0   11.561671
  1   11.342958
  2   11.632240
  3   11.824941
  4   11.372034
  5   11.551291
  6   11.790056
  7   11.657485
  8   11.440671
  9   11.310480


C:\Users\yassi\AppData\Local\Temp\ipykernel_26600\4160864483.py:132: RuntimeWarning: Reproduction stricte non démontrée ; le CSV est nommé « alternative ». Ne pas remplacer la soumission V10 historique à 0.02808.
  warnings.warn(


In [37]:
# 11 — Journal de provenance : enregistrer ce qui a réellement été fait.
# ------------------------------------------------------------------------------
# Le score public 0.02808 est historique et n'est PAS mesuré par ce notebook.
# L'audit a déjà été effectué dans la cellule 10, sans utiliser les prédictions
# V10 originales pour construire une prédiction.

manifest = {
    'modele': 'V10',
    'score_public_historique': 0.02808,
    'soumission_generee': str(OUTPUT_PATH),
    'reproduction_numerique_originale_verifiee': reproduced,
    'mode_prediction': PREDICTION_MODE,
    'source_des_modeles': REGRESSOR_SOURCES,
    'composants_reentraines': RETRAINED_COMPONENTS,
    'explication': (
        'Une restauration par CSV V7/V8 récupère les SORTIES DE TEST historiques '
        'de ExtraTrees : elle ne restaure pas son modèle entraîné. '
        'Une reconstruction complète de l entraînement peut donner des valeurs différentes.'
    ),
    'n_images_train': len(X_train),
    'n_images_test': len(X_test),
    'copies_augmentees': N_COPIES,
    'sigma_bruit': NOISE_SIGMA,
    'translation_max_pixels': MAX_SHIFT,
    'hyperparametres_boosting': HGB_PARAMS,
    'hyperparametres_extra_trees': EXTRA_PARAMS,
    'poids_ensemble': {'boosting': .70, 'extra_trees': .30},
    'versions': {
        'python': os.sys.version.split()[0],
        'numpy': np.__version__, 'pandas': pd.__version__,
        'torch': torch.__version__, 'sklearn': sklearn.__version__,
        'joblib': joblib.__version__,
    },
}
manifest_path = RESULTS_DIR / 'v10_manifest_v2.json'
manifest_path.write_text(json.dumps(manifest, ensure_ascii=False, indent=2), encoding='utf-8')

print('=== TRAÇABILITÉ ===')
print('Mode :', PREDICTION_MODE)
print('Reproduction fidèle V10 :', reproduced)
print('Composants réentraînés :', RETRAINED_COMPONENTS)
print('Fichier de suivi :', manifest_path)
if reproduced is not True:
    print('ATTENTION : conserver le CSV Kaggle original comme soumission de référence.')


=== TRAÇABILITÉ ===
Mode : prédictions historiques récupérées via V8
Reproduction fidèle V10 : False
Composants réentraînés : {'denoiser': False, 'boosting': False, 'extra_trees': False}
Fichier de suivi : results\v10_manifest_v2.json
ATTENTION : conserver le CSV Kaggle original comme soumission de référence.


## D. Validation, résultats et limites

**Protocole de développement (déjà exécuté lors des expériences) :** 1 600 images de développement, cinq plis stratifiés avec 1 280 exemples entraînement / 320 validation. La validation simule le bruit gaussien σ = 0,133942, avec translation maximale 0 ou 10 pixels. Pour V10, chaque pli utilise un débruiteur entraîné et sélectionné **sans les 320 images de validation externe** (partition interne 1 024 / 256).

| Version | MAE CV, translation 10 px | Score public Kaggle |
|---|---:|---:|
| V4 — Boosting42 | 0,034052 | 0,03291 |
| V7 — Boosting42 + ExtraTrees42 | 0,033316 | 0,03158 |
| V8 — Boosting56 + ExtraTrees42 | 0,032510 | 0,03059 |
| **V10 — Boosting70 + ExtraTrees42** | **0,030802** | **0,02808** |

**Attention :** ces nombres sont les résultats d'expériences précédentes et ne sont pas recalculés par la cellule de soumission. Le score Kaggle privé est inconnu. L'ancien score ~0,12958 de V1 malgré sa bonne CV sans translation a motivé la correction du décalage des distributions train/test. Une validation après plusieurs itérations de conception peut surestimer les gains : le protocole ne constitue pas un test indépendant à l'aveugle.

In [38]:
# 12. Tableau d'expériences documentées (valeurs historiques, pas recalculées).
# Les CV de V4/V7/V8/V10 utilisent le scénario avec translation max 10 pixels.
# Ne jamais présenter ces chiffres comme des résultats de ce notebook sans exécuter RUN_CV.
summary_hist=pd.DataFrame([
    {"Modele":"Baseline médiane", "CV_10px":np.nan, "Public":np.nan, "Note":"CV sans augmentation : 0.124952"},
    {"Modele":"Ridge radial", "CV_10px":np.nan, "Public":np.nan, "Note":"CV sans augmentation : 0.052030"},
    {"Modele":"V1 Boosting24", "CV_10px":np.nan, "Public":0.12958, "Note":"CV initiale 0.032648 ; forte translation inconnue"},
    {"Modele":"V4 Boosting42", "CV_10px":0.034052, "Public":0.03291,"Note":"géométrie"},
    {"Modele":"V7 70%B42 +30% Extra42", "CV_10px":0.033316, "Public":0.03158,"Note":"ensemble"},
    {"Modele":"V8 70%B56 +30% Extra42", "CV_10px":0.032510, "Public":0.03059,"Note":"lissage"},
    {"Modele":"V10 70%B70 +30% Extra42", "CV_10px":0.030802, "Public":0.02808,"Note":"U-Net entraîné depuis zéro"}
])
print(summary_hist.to_string(index=False))
summary_hist.to_csv(RESULTS_DIR/"v10_historical_results.csv",index=False)

                 Modele  CV_10px  Public                                              Note
       Baseline médiane      NaN     NaN                   CV sans augmentation : 0.124952
           Ridge radial      NaN     NaN                   CV sans augmentation : 0.052030
          V1 Boosting24      NaN 0.12958 CV initiale 0.032648 ; forte translation inconnue
          V4 Boosting42 0.034052 0.03291                                         géométrie
 V7 70%B42 +30% Extra42 0.033316 0.03158                                          ensemble
 V8 70%B56 +30% Extra42 0.032510 0.03059                                           lissage
V10 70%B70 +30% Extra42 0.030802 0.02808                        U-Net entraîné depuis zéro
